In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parents[2]
sys.path.insert(0, str(project_root / 'src'))


import os
import logging
import yaml
import datetime
import requests
import uuid
import fastf1
import pandas as pd
import polars as pl
import numpy as np
import pyarrow.parquet as pq
import pyarrow as pa
from dataclasses import dataclass
from pathlib import Path

logging.basicConfig(
    encoding='utf-8',
    level=logging.ERROR,
    datefmt='%m/%d/%Y %I:%M:%S %p'
)

pd.set_option('display.max_columns', None)

def yaml_loaders(file_path: str, header: str) -> list:
    with open(file_path, 'r') as yaml_file:
        yaml_data = yaml.safe_load(yaml_file)

    return yaml_data


def make_year_range(start_yr: int, end_yr: int) -> list[int]:
    return list(range(start_yr, end_yr + 1))


def get_yearly_schedule(start_yr: int, end_yr: int) -> pd.DataFrame:

    years = make_year_range(start_yr, end_yr)

    yearly_event_table = []
    for idx, year in enumerate(years):
        tmp_df = fastf1.get_event_schedule(year)
        tmp_df['schedule_year'] = year
        
        yearly_event_table.append(tmp_df)
        
    return pd.concat(yearly_event_table)


def create_uuid_col(df: pd.DataFrame) -> pd.DataFrame:
    df['record_id'] = [str(uuid.uuid7()) for _ in range(len(df))]
    df['record_id'] = df['record_id'].astype('string[pyarrow]')
    return df


def event_schedule_str_type_fix(df: pd.DataFrame) -> pd.DataFrame:

    event_calendar_str: list[str] = [
        'EventName',
        'EventFormat',
        'Location',
        'Country',
        'OfficialEventName',
        'Session1',
        'Session2',
        'Session3',
        'Session4',
        'Session5',
        ]

    df[event_calendar_str] = (
        df[event_calendar_str].astype('string[pyarrow]')
    )

    return df


def event_schedule_tmz_fix(df: pd.DataFrame) -> pd.DataFrame:

    event_calendar_dt: list[str] = [
        'Session1Date',
        'Session2Date',
        'Session3Date',
        'Session4Date',
        'Session5Date'
    ]

    df[event_calendar_dt] = (
        df[event_calendar_dt]
        .apply(lambda s: pd.to_datetime(s, errors='coerce', utc=True).dt.tz_convert('America/New_York'))
    )

    return df


In [2]:
# Getting event data and creating single dataframe.
# Only grabing recent years where regulations were the same or very close.
yearly_schedule_df = get_yearly_schedule(start_yr=2022, end_yr=2025)


req         WARNING 	DEFAULT CACHE ENABLED! (176.0 KB) /home/gfrov/.cache/fastf1


In [ ]:
type(yearly_schedule_df)

In [3]:
event_calendar_str: list[str] = [
    'EventName',
    'EventFormat',
    'Location',
    'Country',
    'OfficialEventName',
    'Session1',
    'Session2',
    'Session3',
    'Session4',
    'Session5',
]

yearly_schedule_df[event_calendar_str] = (
    yearly_schedule_df[event_calendar_str].astype('string[pyarrow]')
)

In [4]:
import hashlib
import uuid

def create_uuid_col(df: pd.DataFrame) -> pd.DataFrame:
    df['record_id'] = [str(uuid.uuid7()) for _ in range(len(df))]
    df['record_id'] = df['record_id'].astype('string[pyarrow]')
    return df

In [5]:
event_calendar_dt: list[str] = [
    'Session1Date',
    'Session2Date',
    'Session3Date',
    'Session4Date',
    'Session5Date'
]

yearly_schedule_df[event_calendar_dt] = (
    yearly_schedule_df[event_calendar_dt]
    .apply(lambda s: pd.to_datetime(s, errors='coerce', utc=True).dt.tz_convert('America/New_York'))
)

In [6]:
yearly_schedule_df = create_uuid_col(yearly_schedule_df)

In [7]:
column_list = yearly_schedule_df.columns.tolist()
new_order = [column_list[-1], column_list[-2]] + column_list[:-2]
yearly_schedule_df = yearly_schedule_df[new_order]

In [25]:
from sqlalchemy import insert
from db.session import engine
from db.models import EventCalendar
from pandas import DataFrame


def clean_df_for_insert(df):
    '''Clean DataFrame for PostgreSQL insertion.'''
    df = df.copy()
    
    for col in df.columns:
        # Replace pandas NaT and NaN with None
        df[col] = df[col].replace({pd.NaT: None, np.nan: None})
        
        # Convert any remaining numpy types to Python native types
        if df[col].dtype == 'datetime64[ns]':
            df[col] = df[col].apply(lambda x: x.to_pydatetime() if pd.notna(x) else None)
    
    return df


def bulk_data_insert_from_df(
        df: DataFrame,
        model_class=EventCalendar,
        chunk_size:int = 10000) -> int:
    
    model_columns = [col.key for col in model_class.__table__.columns]
    rename_map = dict(zip(df.columns, model_columns))
    
    df = df.rename(columns=rename_map)

    df_clean = clean_df_for_insert(df)

    records = df_clean.to_dict(orient='records')
    total = 0

    with engine.begin() as conn:
        for i in range(0, len(records), chunk_size):
            chunk = records[i:i + chunk_size]
            stmt = insert(model_class).values(chunk)
            conn.execute(stmt)
            total += len(chunk)

    return total


In [27]:
from db.base import Base

Base.metadata.create_all(engine)

records_inserted = bulk_data_insert_from_df(yearly_schedule_df)
print(records_inserted)

2025-12-31 15:32:52,422 INFO sqlalchemy.engine.Engine BEGIN (implicit)


INFO:sqlalchemy.engine.Engine:BEGIN (implicit)


2025-12-31 15:32:52,426 INFO sqlalchemy.engine.Engine SELECT pg_catalog.pg_class.relname 
FROM pg_catalog.pg_class JOIN pg_catalog.pg_namespace ON pg_catalog.pg_namespace.oid = pg_catalog.pg_class.relnamespace 
WHERE pg_catalog.pg_class.relname = %(table_name)s::VARCHAR AND pg_catalog.pg_class.relkind = ANY (ARRAY[%(param_1)s::VARCHAR, %(param_2)s::VARCHAR, %(param_3)s::VARCHAR, %(param_4)s::VARCHAR, %(param_5)s::VARCHAR]) AND pg_catalog.pg_table_is_visible(pg_catalog.pg_class.oid) AND pg_catalog.pg_namespace.nspname != %(nspname_1)s::VARCHAR


INFO:sqlalchemy.engine.Engine:SELECT pg_catalog.pg_class.relname 
FROM pg_catalog.pg_class JOIN pg_catalog.pg_namespace ON pg_catalog.pg_namespace.oid = pg_catalog.pg_class.relnamespace 
WHERE pg_catalog.pg_class.relname = %(table_name)s::VARCHAR AND pg_catalog.pg_class.relkind = ANY (ARRAY[%(param_1)s::VARCHAR, %(param_2)s::VARCHAR, %(param_3)s::VARCHAR, %(param_4)s::VARCHAR, %(param_5)s::VARCHAR]) AND pg_catalog.pg_table_is_visible(pg_catalog.pg_class.oid) AND pg_catalog.pg_namespace.nspname != %(nspname_1)s::VARCHAR


2025-12-31 15:32:52,427 INFO sqlalchemy.engine.Engine [generated in 0.00106s] {'table_name': 'event_calendar', 'param_1': 'r', 'param_2': 'p', 'param_3': 'f', 'param_4': 'v', 'param_5': 'm', 'nspname_1': 'pg_catalog'}


...

2025-12-31 15:32:52,501 INFO sqlalchemy.engine.Engine [no key 0.00374s] {'record_id_m0': '019b7690-9cf3-753e-89aa-e924ac22b292', 'schedule_year_m0': 2022, 'round_number_m0': 0, 'round_country_m0': 'Spain', 'round_location_m0': 'Spain', 'round_offical_name_m0': 'FORMULA 1 PRE-SEASON TRACK SESSION 2022', 'round_event_date_m0': Timestamp('2022-02-25 00:00:00'), 'round_event_name_m0': 'Pre-Season Track Session', 'round_event_fmt_m0': 'testing', 'round_type_session_1_m0': 'Practice 1', 'round_date_session_1_m0': Timestamp('2022-02-23 03:00:00-0500', tz='America/New_York'), 'round_date_utc_session_1_m0': Timestamp('2022-02-23 08:00:00'), 'round_type_session_2_m0': 'Practice 2', 'round_date_session_2_m0': Timestamp('2022-02-24 03:00:00-0500', tz='America/New_York'), 'round_date_utc_session_2_m0': Timestamp('2022-02-24 08:00:00'), 'round_type_session_3_m0': 'Practice 3', 'round_date_session_3_m0': Timestamp('2022-02-25 03:00:00-0500', tz='America/New_York'), 'round_date_utc_session_3_m0': Time

INFO:sqlalchemy.engine.Engine:[no key 0.00374s] {'record_id_m0': '019b7690-9cf3-753e-89aa-e924ac22b292', 'schedule_year_m0': 2022, 'round_number_m0': 0, 'round_country_m0': 'Spain', 'round_location_m0': 'Spain', 'round_offical_name_m0': 'FORMULA 1 PRE-SEASON TRACK SESSION 2022', 'round_event_date_m0': Timestamp('2022-02-25 00:00:00'), 'round_event_name_m0': 'Pre-Season Track Session', 'round_event_fmt_m0': 'testing', 'round_type_session_1_m0': 'Practice 1', 'round_date_session_1_m0': Timestamp('2022-02-23 03:00:00-0500', tz='America/New_York'), 'round_date_utc_session_1_m0': Timestamp('2022-02-23 08:00:00'), 'round_type_session_2_m0': 'Practice 2', 'round_date_session_2_m0': Timestamp('2022-02-24 03:00:00-0500', tz='America/New_York'), 'round_date_utc_session_2_m0': Timestamp('2022-02-24 08:00:00'), 'round_type_session_3_m0': 'Practice 3', 'round_date_session_3_m0': Timestamp('2022-02-25 03:00:00-0500', tz='America/New_York'), 'round_date_utc_session_3_m0': Timestamp('2022-02-25 08:00:

2025-12-31 15:32:52,518 INFO sqlalchemy.engine.Engine COMMIT


INFO:sqlalchemy.engine.Engine:COMMIT


97


In [24]:
datetime_cols = [
    'round_event_date',
    'round_date_session_1', 'round_date_utc_session_1',
    'round_date_session_2', 'round_date_utc_session_2',
    'round_date_session_3', 'round_date_utc_session_3',
    'round_date_session_4', 'round_date_utc_session_4',
    'round_date_session_5', 'round_date_utc_session_5',
]

# Check which ones exist in your DataFrame and their types
for col in datetime_cols:
    if col in yearly_schedule_df.columns:
        dtype = yearly_schedule_df[col].dtype
        null_count = yearly_schedule_df[col].isna().sum()
        print(f"{col}: {dtype}, nulls: {null_count}")
        
        # Show a sample value
        sample = yearly_schedule_df[col].dropna().iloc[0] if null_count < len(yearly_schedule_df) else "ALL NULL"
        print(f"  Sample: {sample} (type: {type(sample).__name__})")
        print()